# Change Detection Between Two Frames

This notebook demonstrates **object-level change detection** between two camera frames.

It uses:
- **YOLOv8** for object detection
- **SSIM** (Structural Similarity Index) for change localisation
- **ORB feature matching** for frame alignment

Source: [`change_detection_system.py`](../change_detection_system.py)

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import from the project root

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

from change_detection_system import ChangeDetectionSystem

print('All imports OK')

## Configuration — set your image paths here

In [ ]:
FRAME_BEFORE = '../Image_cc.jpg'    # Reference / earlier frame
FRAME_AFTER  = '../Image2_cc.png'   # Current / later frame
OUTPUT_DIR   = '../outputs'

import os; os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Load and preview input images

In [ ]:
frame_a = cv2.cvtColor(cv2.imread(FRAME_BEFORE), cv2.COLOR_BGR2RGB)
frame_b = cv2.cvtColor(cv2.imread(FRAME_AFTER),  cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(frame_a); axes[0].set_title('BEFORE', fontsize=14, fontweight='bold'); axes[0].axis('off')
axes[1].imshow(frame_b); axes[1].set_title('AFTER',  fontsize=14, fontweight='bold'); axes[1].axis('off')
plt.suptitle('Input Frames', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 2. Initialise the Change Detection System

In [ ]:
detector = ChangeDetectionSystem(model_name='yolov8n.pt')

## 3. Frame Alignment

In [ ]:
fa_bgr = cv2.imread(FRAME_BEFORE)
fb_bgr = cv2.imread(FRAME_AFTER)

fa_aligned, fb_aligned = detector.preprocess_and_align(fa_bgr, fb_bgr)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(fa_aligned, cv2.COLOR_BGR2RGB))
axes[0].set_title('Before (aligned)', fontsize=13); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(fb_aligned, cv2.COLOR_BGR2RGB))
axes[1].set_title('After (aligned)',  fontsize=13); axes[1].axis('off')
plt.suptitle('After ORB + Homography Alignment', fontsize=15)
plt.tight_layout(); plt.show()

## 4. SSIM Change Map

In [ ]:
change_map = detector.compute_change_map(fa_aligned, fb_aligned)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(cv2.cvtColor(fa_aligned, cv2.COLOR_BGR2RGB)); axes[0].set_title('Before'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(fb_aligned, cv2.COLOR_BGR2RGB)); axes[1].set_title('After');  axes[1].axis('off')
axes[2].imshow(change_map, cmap='hot'); axes[2].set_title('SSIM Change Map'); axes[2].axis('off')
plt.suptitle('Change Localisation via SSIM', fontsize=15)
plt.tight_layout(); plt.show()

changed_pct = np.sum(change_map > 0) / change_map.size * 100
print(f'Changed area: {changed_pct:.2f}% of the frame')

## 5. Object Detection (YOLOv8)

In [ ]:
detections_a = detector.detect_objects(fa_aligned)
detections_b = detector.detect_objects(fb_aligned)

print(f'Objects detected in BEFORE frame: {len(detections_a)}')
for d in detections_a:
    print(f"  {d['class']:20s}  conf={d['confidence']:.2f}  bbox={d['bbox']}")

print(f'\nObjects detected in AFTER frame: {len(detections_b)}')
for d in detections_b:
    print(f"  {d['class']:20s}  conf={d['confidence']:.2f}  bbox={d['bbox']}")

## 6. Change Classification

In [ ]:
changes = detector.match_objects(detections_a, detections_b)

print('ADDED objects:')
for c in changes['added']:   print(f"  + {c['object']}")
print('\nREMOVED objects:')
for c in changes['removed']: print(f"  - {c['object']}")
print('\nMOVED objects:')
for c in changes['moved']:   print(f"  ~ {c['object']}  direction={c['direction']}  distance={c['distance']:.1f}px")

## 7. Change Summary — Natural Language Description

In [ ]:
description = detector.generate_description(changes)

print('='*60)
print('CHANGE SUMMARY')
print('='*60)
print(description)

## 8. Annotated Output Visualisation

In [ ]:
vis_a, vis_b = detector.visualize_results(fa_aligned, fb_aligned,
                                           detections_a, detections_b, changes)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(cv2.cvtColor(vis_a, cv2.COLOR_BGR2RGB))
axes[0].set_title('BEFORE – Blue=detected, Red=removed', fontsize=12); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(vis_b, cv2.COLOR_BGR2RGB))
axes[1].set_title('AFTER – Green=added, Yellow=moved, Blue=unchanged', fontsize=12); axes[1].axis('off')

legend = [
    mpatches.Patch(color='lime',   label='Added'),
    mpatches.Patch(color='yellow', label='Moved'),
    mpatches.Patch(color='blue',   label='Unchanged'),
    mpatches.Patch(color='red',    label='Removed (before frame)'),
]
plt.legend(handles=legend, loc='upper center', bbox_to_anchor=(0, -0.05),
           ncol=4, fontsize=11)
plt.suptitle('Change Detection Result', fontsize=16)
plt.tight_layout(); plt.show()

import cv2
cv2.imwrite(f'{OUTPUT_DIR}/object_change_before.jpg', vis_a)
cv2.imwrite(f'{OUTPUT_DIR}/object_change_after.jpg',  vis_b)
print('Saved to outputs/')

## Summary Table

In [ ]:
import pandas as pd

rows = []
for c in changes['added']:   rows.append({'Change Type': 'Added',   'Object': c['object'], 'Details': str(c['bbox'])})
for c in changes['removed']: rows.append({'Change Type': 'Removed', 'Object': c['object'], 'Details': str(c['bbox'])})
for c in changes['moved']:   rows.append({'Change Type': 'Moved',   'Object': c['object'], 'Details': f"direction={c['direction']}, dist={c['distance']:.0f}px"})

if rows:
    df = pd.DataFrame(rows)
    display(df.style.set_caption('Detected Changes').set_table_styles(
        [{'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold')]}]
    ))
else:
    print('No object-level changes detected.')